# Ma Soi — Behavior Cloning on Colab

Train a learned policy from heuristic-bot self-play data (§17).

## Before opening this notebook

On your machine, generate + validate + encode the dataset:

```powershell
npx tsx apps/server/scripts/selfplay.ts --games 10000 --players 8 --preset --defense --seed bc --trajectories .tmp/bc --trace-games 10000 --no-jitter --quiet
npm run ai:validate-dataset -- .tmp/bc/trajectories.jsonl
npm run ai:encode -- --in .tmp/bc/trajectories.jsonl --out .tmp/bc/enc
```

The validator must print `dataset SẠCH`. `--no-jitter` is required: without it the
teacher is non-deterministic and the agreement ceiling drops to ~0.60.

Then zip and upload to Drive:

```powershell
Compress-Archive -Path ai-training\masoi_training, .tmp\bc\enc -DestinationPath .tmp\bc\bc-package.zip -Force
```

The zip holds `masoi_training/` (training code) + `enc/` (`.bin` tensors). Skip the
6 GB JSONL — encoded tensors compress to ~84 MB.

Boundary (§39): game rules + encoder run on your machine in TypeScript. Colab only sees numbers.

## 0. Check GPU

Colab sometimes gives you a CPU runtime silently. No GPU: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# torch moved the ONNX exporter to a separate package. Install here (not in the
# train cell): train_bc runs in a subprocess, so this takes effect without a restart.
%pip install -q onnxscript

import torch

print("torch", torch.__version__, "| cuda build", torch.version.cuda)
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU  {name}  ({vram:.1f} GB)")
else:
    print("NO GPU - Runtime > Change runtime type > T4 GPU, then re-run this cell.")

## 1. Load data from Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# Change to wherever you put the file on Drive.
ZIP = "/content/drive/MyDrive/bc-package.zip"

!rm -rf /content/bc && mkdir -p /content/bc
!unzip -q "$ZIP" -d /content/bc
!ls /content/bc /content/bc/enc

## 2. Validate data before training

Three questions before spending GPU time:

1. Do files match `meta.json` — a truncated file still reshapes "fine" and shifts
   every label by a row.
2. Do train/val/test sizes sum to the full set (§15)?
3. **Is every label LEGAL under its own mask** — otherwise the dataset teaches
   out-of-rule moves?

In [ ]:
import gc
import glob
import os
import sys

sys.path.insert(0, "/content/bc")
from masoi_training.data import action_distribution, load

# Auto-find the dataset dir: the full package names it `enc`, the slim one `enc-slim`.
found = [p for p in sorted(glob.glob("/content/bc/*")) if os.path.isfile(os.path.join(p, "meta.json"))]
if len(found) != 1:
    raise SystemExit(f"need exactly 1 dir with meta.json under /content/bc, saw: {found}")
DATA = found[0]
print("dataset dir:", DATA)


def check(path):
    """Check inside a function so numpy arrays die on return.

    `load()` puts ALL tensors in RAM (~3.1 GB for the full package). If `data`
    lived at notebook scope, the train cell's subprocess would load ANOTHER copy
    on top of its own train/val/test splits - past Colab free RAM, killed with
    exit -9 (SIGKILL) and an empty log. Seen it happen. Return numbers, not arrays.
    """
    data = load(path)  # throws if a file mismatches meta.json

    sizes = {name: len(data.split(name)) for name in ("train", "validation", "test")}
    assert sum(sizes.values()) == len(data)
    assert data.masks[range(len(data)), data.actions].all(), "labels point at ILLEGAL actions"
    assert set(data.rewards.tolist()) <= {-1.0, 1.0}

    # `optimal.u8.bin` is what `agreementTieAware` relies on. Without it the §17
    # headline metric is null and only `agreement` (which penalizes tied moves) remains.
    assert data.optimal is not None, "missing optimal.u8.bin - re-run ai:encode"

    return {
        "version": data.meta.get("datasetVersion"),
        "commit": (data.meta.get("gitCommit") or "")[:8],
        "rows": len(data),
        "obs": data.obs_size,
        "actions": data.action_size,
        "games": data.meta.get("games"),
        "sizes": sizes,
        "classes": len(action_distribution(data)),
        "scores": data.scores is not None,
    }


info = check(DATA)
gc.collect()

print("dataset  ", info["version"], "commit", info["commit"])
print("rows     ", info["rows"], "| obs", info["obs"], "| actions", info["actions"])
print("games    ", info["games"])
print("split    ", info["sizes"])
print("action classes (§43):", info["classes"])
print("scores   ", "present (only used with --distill-alpha > 0)" if info["scores"] else "absent - not needed by default")
print("\nOK - ready to train. RAM was released before the train cell.")

## 3. Smoke test: 2 epochs

Catch config errors here in 30 seconds instead of after a full GPU run.

In [ ]:
import subprocess
import sys

done = subprocess.run(
    [sys.executable, "-m", "masoi_training.train_bc",
     "--data", DATA, "--out", "/content/smoke", "--epochs", "2"],
    cwd="/content/bc",
)
if done.returncode != 0:
    raise SystemExit(f"smoke test failed (exit {done.returncode}) - read the log above")
print("smoke OK")

## 4. Full train

`--batch-size 512` is the script default every other report in this repo uses — keep it
so `metrics.json` stays comparable. `1024` is faster (fewer steps, less Python overhead)
at slightly coarser gradients.

Small net — 95,949 params, 16.9 TFLOP for all 40 epochs — bottlenecked by per-step
overhead, not GPU flops. Don't expect a T4 to beat a low-end discrete card by much.

In [ ]:
import subprocess
import sys
import time

# subprocess, NOT `!`: a failed `!` lets the notebook continue, and the results
# cell below then throws FileNotFoundError on metrics.json - wrong place blamed.
# A non-zero exit stops right here with the real log.
CMD = [
    sys.executable, "-m", "masoi_training.train_bc",
    "--data", DATA,
    "--out", "/content/model-v001",
    "--epochs", "40",
    "--batch-size", "512",
    "--lr", "1e-3",
    "--hidden", "128",
    "--seed", "12345",
    "--model-id", "policy-v001",
]

!rm -rf /content/model-v001
start = time.time()
done = subprocess.run(CMD, cwd="/content/bc")
print(f"\ntotal {time.time() - start:.0f}s  | exit {done.returncode}")
if done.returncode != 0:
    raise SystemExit(
        f"train_bc failed (exit {done.returncode}). Read the log above.\n"
        "On OOM Colab kills the process and the log may be empty: see 'Out of RAM' at the end."
    )

## 5. Read the results

The deciding number is **`metrics.test.agreementTieAware`** — does the model pick a
TIED-FOR-BEST move vs the teacher on unseen games?

Not `agreement`: it penalizes tied moves the teacher broke by raw id — info §9 hides
from the observation, so no model can learn it. Still printed for old-table comparison.

And don't judge by loss. §17: no baseline replication, no RL. Falling loss with flat
agreement means the model is learning the majority class, not the game.

In [ ]:
import json
import pathlib

path = pathlib.Path("/content/model-v001/metrics.json")
if not path.exists():
    raise SystemExit(
        f"no {path} - cell 4 (train) did NOT finish or failed.\n"
        "Go back to cell 4, read the final 'exit ...' line. Don't edit this cell."
    )
report = json.loads(path.read_text())
m = report["metrics"]

print(f"{'':<11} {'tieAware':>9} {'agreement':>10} {'top-2':>8}")
for name in ("train", "validation", "test"):
    e = m[name]
    print(f"{name:<11} {e.get('agreementTieAware'):>9} {e.get('agreement'):>10} {e.get('top2Agreement'):>8}")

# Diagnose on the SAME metric used for conclusions. Mixing an `agreement` gap with
# `agreementTieAware` conclusions mixes two scales.
gap = m["train"]["agreementTieAware"] - m["validation"]["agreementTieAware"]
print(f"\ntrain - val = {gap:+.4f}  ->", "OVERFIT" if gap > 0.05 else "no overfit")
print("best epoch:", report["bestEpoch"], "/", report["trainingConfig"]["epochs"])

print("\nBy decision type (worst first):")
for kind, score in sorted(m["test"].get("agreementByDecision", {}).items(), key=lambda kv: kv[1]):
    print(f"  {kind:<13} {score}")

print("\nBy role (worst first):")
for role, score in sorted(m["test"].get("agreementByRole", {}).items(), key=lambda kv: kv[1]):
    print(f"  {role:<18} {score}")

print("\nCalibration (ECE - lower is better, > 0.1 is overconfident):")
print(f"  test ece: {m['test'].get('ece')}")
for kind, score in sorted(m["test"].get("eceByDecision", {}).items(), key=lambda kv: kv[1]):
    print(f"  {kind:<13} {score}")

In [ ]:
import matplotlib.pyplot as plt

history = report["history"]
epochs = [row["epoch"] for row in history]

figure, left = plt.subplots(figsize=(8, 4))
left.plot(epochs, [row["trainLoss"] for row in history], label="train loss")
left.set_xlabel("epoch")
left.set_ylabel("loss")

right = left.twinx()
right.plot(
    epochs,
    [r["val_agreementTieAware"] if r.get("val_agreementTieAware") is not None else r.get("val_agreement") for r in history],
    color="tab:orange",
    label="val agreement",
)
right.set_ylabel("agreement")

figure.legend(loc="upper right")
plt.title("Behavior cloning: falling loss is NOT enough, agreement must rise")
plt.show()

## 6. Save the model to Drive

The TypeScript runtime reads **`model.weights.json`** (`masoi-mlp-1/2`), NOT `model.onnx`:
`game-engine` must stay pure and every bot decision is synchronous, while
`onnxruntime-node` is async-only (see `masoi_training/export.py`). The `.onnx` is for
inspection or other tooling.

Keep `metrics.json` next to `model.weights.json` — it carries `modelId`, `gitCommit`,
`datasetVersion`, `trainingSeed`. A model that can't be traced to its dataset and commit
isn't reproducible (§46).

Back on your machine, drop the folder into `.tmp/bc/model-v001/` and benchmark first:

```bash
npm run ai:benchmark -- --model .tmp/bc/model-v001/model.weights.json --setups baseline,village,wolves,all,teacher
```

NEVER overwrite `apps/server/assets/models/` — that's the model shipping in the image.

In [ ]:
!mkdir -p /content/drive/MyDrive/masoi-models
!cp -r /content/model-v001 /content/drive/MyDrive/masoi-models/
!ls -lh /content/drive/MyDrive/masoi-models/model-v001

# Or download directly (small model, a few MB):
# from google.colab import files
# !cd /content && zip -qr model-v001.zip model-v001
# files.download("/content/model-v001.zip")

## Reading the numbers

Read `test.agreementTieAware`, against the **dataset ceiling**, not 1.0.

The ceiling is the "agreement ceiling (§17)" line `npm run ai:validate-dataset` prints.
For the 10,002-game dataset of 2026-09-16 it is **0.984**. The rest is moves the teacher
broke with info absent from the observation; no policy can learn that part.

| tieAware / ceiling | Meaning | Do |
|---|---|---|
| < 0.35 | Barely learned anything | See below |
| 0.35 – 0.70 | Trend learned, bot not replicated | Raise `--epochs` / `--hidden` |
| > 0.75 | Baseline replicated well | §17 cleared, RL is on the table |

### Agreement stalled?

Look at `agreementByDecision` **before** touching the model. `SPEECH` has no labels in
this action space — 862,075 / 1,928,813 rows are "unlabeled" for exactly that reason —
so if one decision type drags the average, that's where to look, not `--hidden`.

Raise `--hidden` only when train ≈ val and both are low (underfit). If train ≫ val,
a bigger model feeds the disease.

Don't seed-shop. Changing `--seed` tests stability; if two seeds disagree widely,
neither number concludes anything.

## Out of RAM (exit -9)

`exit -9` is SIGKILL: Linux killed the process for RAM. The log is usually EMPTY,
so it never names the cause.

`masoi_training.data.load()` loads ALL tensors into RAM, then `split()` copies three
more. On the 10,002-game dataset:

| | full package | slim package |
|---|---|---|
| full | 3.08 GB | 2.21 GB |
| + train / val / test | 2.12 + 0.48 + 0.48 | 1.52 + 0.34 + 0.34 |
| **one process** | **6.16 GB** | **4.41 GB** |

Colab free has ~12.7 GB. One process fits. **Two** don't — and that's the trap: if the
check cell keeps `data` alive, the notebook holds 3.08 GB while the train subprocess
loads 6.16 GB more. The check cell above already returns RAM right after checking.

Still OOM? Drop `scores.f32.bin` — only used with `--distill-alpha > 0` (default 0):

```powershell
Remove-Item .tmp\bc\enc\scores.f32.bin
Compress-Archive -Path ai-training\masoi_training, .tmp\bc\enc -DestinationPath .tmp\bc\bc-package-slim.zip -Force
```

`data.py` treats the file as optional, nothing else to change.

Measured (full dataset, process RSS): 0.03 GB before check → 3.12 GB inside (5.25 GB peak
while splitting) → 0.03 GB after return. RAM fully reclaimed before train runs.